# BRAIN-HYBRID — Cerveau Artificiel

**Qwen3-VL-8B** (gelé, Q8) + 4 modules CfC+SNN + STDP + Hippocampe SDM

Chaque cellule est autonome. Si le runtime redémarre, relancez depuis la cellule 1.

In [ ]:
#@title 1. Installation + Chargement modèle
!pip install transformers ncps snntorch accelerate bitsandbytes qwen-vl-utils tqdm -q
!git clone -b refactor/predictive-coding-20260320-143000-001 https://github.com/Asurelia/SDNC.git /content/sdnc 2>/dev/null || (cd /content/sdnc && git pull --ff-only)

import os, sys, torch
os.chdir('/content/sdnc')
if '/content/sdnc' not in sys.path:
    sys.path.insert(0, '/content/sdnc')

from brain_hybrid.model import BrainHybridModel
from brain_hybrid.config import BrainConfig

config = BrainConfig()
model = BrainHybridModel(config)

print(f'\nGPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.memory_allocated(0)/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Modèle prêt')

In [ ]:
#@title 2. Tests d'intégration
import torch

reps = model.llm.get_layer_representations('Le chat mange', layers=[9, 36])
sim = torch.nn.functional.cosine_similarity(
    reps[0].float().mean(dim=1), reps[1].float().mean(dim=1)
).item()
print(f'Hidden states cosine sim : {sim:.4f} — {"PASS" if sim < 0.95 else "FAIL"}')

result = model.forward('Test de fonctionnement', learn=True)
print(f'Forward OK — erreurs: {[f"{e:.3f}" for e in result["prediction_errors"]]}')
print(f'Réponse : {result["response"][:100]}')
print(f'VRAM : {torch.cuda.memory_allocated(0)/1e9:.1f} GB')
print('TOUS LES TESTS PASSENT')

In [ ]:
#@title 3. Apprentissage continu (100 steps)
from brain_hybrid.eval.continual_test import run_continual_test

results = run_continual_test(model, n_steps=100, verbose=True)

In [ ]:
#@title 4. Visualisation
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(model.error_history, alpha=0.7)
axes[0].set_title('Erreur de prédiction')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Erreur')
axes[0].grid(True, alpha=0.3)

dop_vals = [s.dopamine_signal for s in model.stdp_learners]
axes[1].bar([f'M{i}' for i in range(len(dop_vals))], dop_vals)
axes[1].set_title('Dopamine par module')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Erreur initiale : {model.error_history[0]:.4f}')
print(f'Erreur finale   : {model.error_history[-1]:.4f}')
print(f'Amélioration    : {(1 - model.error_history[-1]/model.error_history[0])*100:.1f}%')
print(f'Hippocampe      : {model.hippocampus.stats()}')

In [ ]:
#@title 5. Test mémoire épisodique
model.forward('Mon chat Luna est roux et adore la laine', learn=True)
model.forward('Luna joue avec des balles de laine rouge', learn=True)

recalled = model.remember('chat Luna')
print(f'Souvenir récupéré — norme : {recalled.norm().item():.3f}')
print(f'Stats hippocampe : {model.hippocampus.stats()}')

In [ ]:
#@title 6. Export modèle final vers Google Drive
import os, torch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_PATH = '/content/drive/MyDrive/brain_hybrid/'
export_path = DRIVE_PATH + 'releases/'
os.makedirs(export_path, exist_ok=True)

export_file = export_path + f'brain_final_step{model.step_count}.pt'

torch.save({
    'brain_modules': model.brain_modules.state_dict(),
    'hippocampus_contents': model.hippocampus.contents,
    'hippocampus_counts': model.hippocampus.access_counts,
    'hippocampus_metadata': model.hippocampus.metadata,
    'global_state': model.global_state,
    'error_history': model.error_history,
    'step_count': model.step_count,
    'stdp_dopamine': [s.dopamine_signal for s in model.stdp_learners],
    'config': model.config,
}, export_file)

size_mb = Path(export_file).stat().st_size / 1e6
print(f'Export terminé : brain_final_step{model.step_count}.pt')
print(f'Taille : {size_mb:.1f} MB')
print(f'Emplacement : {export_file}')